### Day5. 다중 선형 회귀

**Advertising 데이터셋**
- TV : TV 광고비(천 달러 단위)
- radio : 라디오 광고비
- newspaper : 신문 광고비
- sales : 제품 판매량 (단위: 천 개)
- 200개 데이터

In [2]:
# 데이터 이해
import pandas as pd
pd.set_option('display.width', 120)
url = "https://www.statlearning.com/s/Advertising.csv"
df = pd.read_csv(url, index_col=0)
print(df.head(3), df.shape, sep="\n")

      TV  radio  newspaper  sales
1  230.1   37.8       69.2   22.1
2   44.5   39.3       45.1   10.4
3   17.2   45.9       69.3    9.3
(200, 4)


다음과 같은 다중선형회귀 모형을 사용한 회귀모델을 만들고 결과를 확인합니다.
- Advertising.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 합니다.
- 종속변수 : sales
- 독립변수 : sales를 제외한 모든 변수
- 처음부터 순서대로 150개 데이터를 train, 나머지 50개를 test로 사용합니다.
- 모델 생성시 train 데이터를 사용합니다.

In [4]:
#5-1) 위의 조건에 맞게 OLS / OLS.from_formula 모델을 생성하고 summary를 출력한다.
train = df[:150]
test = df[150:]
# print(train.shape, test.shape) # (150, 4) (50, 4)
from statsmodels.api import OLS
formula = "sales ~ TV + radio + newspaper"
model = OLS.from_formula(formula, train).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.896
Model:                            OLS   Adj. R-squared:                  0.894
Method:                 Least Squares   F-statistic:                     418.2
Date:                Sat, 15 Nov 2025   Prob (F-statistic):           1.90e-71
Time:                        13:50:40   Log-Likelihood:                -291.75
No. Observations:                 150   AIC:                             591.5
Df Residuals:                     146   BIC:                             603.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.0298      0.371      8.172      0.0

In [5]:
#5-2) 위에서 생성한 모델에서 적합된 모형 결정계수를 구해 반올림하여 소수점 아래 3자리까지 출력한다.
print(model.rsquared.round(3))

0.896


In [7]:
#5-3) 위에서 생성한 모델에서 통계적으로 유의미한 변수는 몇 개인가?
# 유의수준 : 5%
print(sum(model.pvalues[1:] < 0.05))

2


In [9]:
#5-4) 위의 모델에서 통계적으로 유의한 변수들과 TV와 Radio의 교호작용항을 사용하여
# 새롭게 모델링하여 model2를 생성한다.
s = model.pvalues[1:] < 0.05
cols = s[s].index
formula2 = "sales ~ TV + radio + TV:radio"
model2 = OLS.from_formula(formula2, train).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.967
Model:                            OLS   Adj. R-squared:                  0.966
Method:                 Least Squares   F-statistic:                     1408.
Date:                Sat, 15 Nov 2025   Prob (F-statistic):          1.65e-107
Time:                        13:53:21   Log-Likelihood:                -206.40
No. Observations:                 150   AIC:                             420.8
Df Residuals:                     146   BIC:                             432.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      6.9633      0.300     23.232      0.0

In [10]:
#5-5) 다음의 값을 사용하여 예측값을 구해, 반올림하여 소수점아래 4자리까지 출력한다.
# TV = 170.5, radio=13.2, newspaper=43.2
data = pd.DataFrame({'TV': [170.5], 'radio': [13.2], 'newspaper': [43.2]})
print(round(model2.predict(data)[0], 4))

12.8629


In [ ]:
#5-6) 다음의 모델은 통계적 유의미해석에 사용하는 가설에서 모델은 귀무가설을 기각하는가 채택하는가?
# 귀무가설 : 모든 독립변수가 종속변수에 영향을 주지 않는다.
# 대립가설 : 적어도 하나의 독립변수가 종속변수에 영향을 준다.
# 신뢰수준 : 95%
# for pvalue in model2.pvalues[1:]:
#     if pvalue < 0.05:
#         print("기각")
#     else:
#         print("채택")
# print("기각" for param in model2.params[1:] if model2.pvalues[param] < 0.05 else "채택")
for index in model2.params[1:].index:
    if model2.pvalues[index] < 0.05:
        print(f"{index}: 기각")
    else:
        print(f"{index}: 채택")


TV: 기각
radio: 기각
TV:radio: 기각


In [32]:
# 5-7) 모델에서 가장 영향력 있는 변수의 t-value를 구해, 반올림하여 소수점 아래 3자리까지 출력한다.
parameter = model2.tvalues[1:].idxmax()
print(round(model2.tvalues[parameter], 3))

17.599


In [35]:
# 5-8) 모델에서 가장 유의미한 변수의 회귀계수를 구해, 반올림하여 소수점 아래 4자리까지 출력한다.
parameter = model2.pvalues[1:].idxmin()
print(round(model2.pvalues[parameter], 4))

0.0


In [46]:
# 5-9) train 데이터를 사용하여 해당 모델의 예측값과 실제값의 피어슨(pearson) 상관계수를 구하여라.
# 결과는 반올림하여 소수점 아래 3자리까지 출력한다.
x_train = train.drop(columns=['sales'])
x_true = train['sales']
x_pred = model2.predict(x_train).round(1)
# print(x_true)
# print(x_pred)
print(x_true.corr(x_pred).round(3))


0.983


In [47]:
# 5-10) train 데이터를 사용하여 해당 모델의 예측값과 실제값의 스피어만(spearman) 상관계수를 구하여라.
# 결과는 반올림하여 소수점 아래 3자리까지 출력한다.
print(x_true.corr(x_pred, method="spearman").round(3))

0.994


In [51]:
# 5-11) test 데이터를 사용하여 rmse를 구해, 반올림하여 소수점 아래 4자리까지 출력한다
from sklearn.metrics import mean_squared_error as MSE
import numpy as np
y_train = test.drop(columns=['sales'])
y_test = test['sales']
y_pred = model2.predict(y_train)
print(round(MSE(y_test, y_pred) ** 0.5, 4))

0.8665


In [54]:
# 5-12) train 데이터를 사용하여 잔차를 구하고, 잔차의 IQR을 구해, 반올림하여 소수점 아래 4자리까지 출력한다.
residual = x_true - x_pred
Q1, Q3 = residual.quantile([0.25, 0.75])
print(round(Q3 - Q1, 4))

1.0


In [71]:
#5-13) 통계적으로 가장 유의하지 않은 변수의 표준오차(Standard Error)를 소수점 아래 4자리까지 출력한다.
parameter = model2.pvalues.idxmax()
print(model2.stderr[parameter])


AttributeError: 'OLSResults' object has no attribute 'stderr'